# 01 — Workout/Fitness videos to pose landmarks

This notebook converts the **378 selected videos** from the Kaggle Workout/Fitness dataset into compact `(time, 17 joints, 4 values)` arrays.

Each joint stores normalized `x, y, z` plus MediaPipe visibility. The final cell verifies the count, creates a ZIP, and immediately opens the browser download. Run the cells from top to bottom in a fresh Colab runtime.


In [ ]:
%pip install -q --upgrade kagglehub mediapipe==0.10.35

In [ ]:
from pathlib import Path
import hashlib
import os
import shutil
import urllib.request

import cv2
import kagglehub
import mediapipe as mp
import numpy as np
import pandas as pd
from google.colab import files
from tqdm.auto import tqdm

SEED = 42
TARGET_FPS = 15
OUTPUT_ROOT = Path("/content/calorie_workoutfitness_landmarks_v1")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("MediaPipe:", mp.__version__)
print("Free temporary disk:")
os.system("df -h /content | tail -1")

In [ ]:
# KaggleHub downloads once per runtime and returns the real cache path.
DATASET_ROOT = Path(kagglehub.dataset_download("hasyimabdillah/workoutfitness-video"))

CLASS_MAP = {
    "barbell biceps curl": "bicep_curl",
    "bench press": "bench_press",
    "deadlift": "deadlift",
    "lat pulldown": "lat_pulldown",
    "lateral raise": "lateral_raise",
    "plank": "plank",
    "pull Up": "pull_up",
    "push-up": "push_up",
    "shoulder press": "shoulder_press",
    "squat": "squat",
}
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".webm"}

selected = []
for source_name, canonical_name in CLASS_MAP.items():
    videos = sorted(
        path for path in (DATASET_ROOT / source_name).rglob("*")
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    )
    selected.extend((path, source_name, canonical_name) for path in videos)
    print(f"{source_name:25s} -> {canonical_name:18s}: {len(videos)}")

print("Total selected:", len(selected))
assert len(selected) == 378, 'Expected 378 videos. Check the mounted dataset version/path.'

In [ ]:
# Download Google's official MediaPipe Full pose model.
MODEL_PATH = Path("/content/pose_landmarker_full.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_full/float16/latest/pose_landmarker_full.task"
)
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Pose model ready:", MODEL_PATH.exists(), MODEL_PATH)

## Landmark extraction logic

- MediaPipe finds 33 body landmarks in each sampled video frame.
- We retain the 17 COCO-compatible joints shared with MM-Fit.
- Hip midpoint becomes the origin and torso size becomes the scale, reducing camera-position and body-size effects.
- Missing detections are interpolated across time. Detection rate is stored so low-quality clips can be rejected later.


In [ ]:
KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    shoulder_width = np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])
    torso_length = np.linalg.norm(shoulder_center - hip_center)
    scale = max(float(shoulder_width), float(torso_length), 1e-6)
    xyz = (xyz - hip_center) / scale
    return np.concatenate([xyz, frame[:, 3:4]], axis=1).astype(np.float32)


def fill_missing(sequence):
    shape = sequence.shape
    flat = sequence.reshape(shape[0], -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction="both").fillna(0).to_numpy()
    return filled.reshape(shape).astype(np.float32)


def file_fingerprint(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    size = path.stat().st_size
    digest.update(str(size).encode())
    with path.open("rb") as handle:
        digest.update(handle.read(block_size))
        if size > block_size:
            handle.seek(max(0, size - block_size))
            digest.update(handle.read(block_size))
    return digest.hexdigest()


def extract_video(video_path):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        return None, {"status": "open_failed"}

    source_fps = float(capture.get(cv2.CAP_PROP_FPS) or 0) or 30.0
    sample_interval = 1.0 / TARGET_FPS
    next_sample_s = 0.0
    frame_index = 0
    poses, timestamps = [], []
    detected = 0

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )

    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            time_s = frame_index / source_fps
            frame_index += 1
            if time_s + 1e-9 < next_sample_s:
                continue
            next_sample_s += sample_interval

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(image, int(round(time_s * 1000)))
            if result.pose_landmarks:
                points = result.pose_landmarks[0]
                pose = np.asarray([
                    [points[i].x, points[i].y, points[i].z, float(points[i].visibility or 0.0)]
                    for i in KEEP_17
                ], dtype=np.float32)
                poses.append(normalize_pose(pose))
                detected += 1
            else:
                poses.append(np.full((17, 4), np.nan, dtype=np.float32))
            timestamps.append(time_s)

    capture.release()
    if not poses:
        return None, {"status": "no_frames"}

    sequence = fill_missing(np.stack(poses))
    detection_rate = detected / len(sequence)
    return sequence, {
        "status": "ok" if detection_rate >= 0.5 else "low_detection",
        "source_fps": source_fps,
        "sampled_frames": len(sequence),
        "detection_rate": detection_rate,
        "duration_s": timestamps[-1] if timestamps else 0.0,
        "timestamps_s": np.asarray(timestamps, dtype=np.float32),
    }

In [ ]:
# Full extraction. The manifest is rewritten after every clip.
MANIFEST_PATH = OUTPUT_ROOT / "manifest.csv"
rows = []

for video_path, source_class, canonical_class in tqdm(selected, desc="Extracting poses"):
    relative_path = video_path.relative_to(DATASET_ROOT)
    clip_id = hashlib.sha1(str(relative_path).encode()).hexdigest()[:16]
    landmarks, info = extract_video(video_path)
    output_path = OUTPUT_ROOT / canonical_class / f"{clip_id}.npz"

    if landmarks is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        timestamps_s = info.pop("timestamps_s")
        np.savez_compressed(
            output_path,
            landmarks=landmarks,
            timestamps_s=timestamps_s,
            label=canonical_class,
            source_dataset="workoutfitness-video",
            original_path=str(relative_path),
            target_fps=TARGET_FPS,
        )

    rows.append({
        "clip_id": clip_id,
        "source_dataset": "workoutfitness-video",
        "person_id": "",
        "group_id": f"workoutfitness-video:{clip_id}",
        "source_class": source_class,
        "canonical_class": canonical_class,
        "original_path": str(relative_path),
        "output_path": str(output_path.relative_to(OUTPUT_ROOT)) if landmarks is not None else "",
        "fingerprint": file_fingerprint(video_path),
        **{key: value for key, value in info.items() if key != "timestamps_s"},
    })
    pd.DataFrame(rows).to_csv(MANIFEST_PATH, index=False)

manifest = pd.DataFrame(rows)
manifest.tail()

In [ ]:
# Verify before downloading. This prevents another empty/lost landmark package.
npz_files = list(OUTPUT_ROOT.rglob("*.npz"))
print("Saved landmark files:", len(npz_files))
print("Status counts:")
print(manifest["status"].value_counts(dropna=False))
print()
print("Detection by class:")
print(manifest.groupby("canonical_class")["detection_rate"].agg(["count", "mean", "min"]).round(3))

assert len(npz_files) == 378, "Extraction is incomplete; inspect failed rows before continuing."
assert manifest["output_path"].ne("").all(), "At least one video has no saved landmark file."

zip_path = shutil.make_archive(
    "/content/calorie_workoutfitness_landmarks_v1", "zip", root_dir=OUTPUT_ROOT
)
print()
print("ZIP ready:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)